In this lab, I combined two datasets, one with NBA players and their performance statistics, and another with the same players and their salaries. The goal of this lab was to find 3 players who overperformed for their salary, 3 players who underperformed for their salary, and 3 players who were relatively average in their performance for their salary. To find these players, I looked at all of the statistics in the dataset. I dropped any statistic that I thought would not be important in identifying a player, such as position, since we are looking for players of any position. Then, I standardized the numeric values in the dataset to make it easier to identify outliers and conduct clustering techniques. I thought about what attributes might be easily visualized to display these clusters, so I decided to take offensive and defensive statistics to find all-around players. These statistics include points scored, defensive rebounds, and assists. The graph below, which will render in a separate browser, shows all of the players in the dataset sorted into 3 clusters, those who perform poorly in all 3, those who performed fairly average, and those who performed well in all three. I decided on three clusters because of the plot shown below that will render in another browser. This plot shows that the optimal number of clusters in this situation is 3. I then created a column named "Cluster" and added it to the dataframe. This column assigns a player a 0, 1, or 2, depending on whether or not they performed badly in defensive rebounds, total points scored, and assists per game. Players who perform poorly are assigned a 0. Those who perform well are assigned a 1. Players who performed close to the mean in these categories are assigned a 2. These numbers are also assigned to players based on their salary. If they are paid too much for how they are performing, they are given a 0. If they are under paid for their good performances, they are given a 1. If they are paid a good amount for their average performances, they are given a 2. I then printed the first three players in each of these clusters to suggest, or not suggest, to Mr. Rooney. For example, the bad players that I recommended not scouting and signing are Santi Aldama, Nickeil Alexander-Walker, and Jose Alvarado because they are not performing as expected, and are therefore overpaid. The 3 good players that I recommend scouting and potentially trying to sign are Grayson Allen, Cole Anthony, OG Anunoby because they are performing better than expected and are therefore underpaid for their contributions to their respective teams. The 3 players that I would consider keeping an eye out for, but not signing yet are Precious Achiuwa, Steven Adams, and Bam Adebayo because these players are performing just as expected. While they are not superstars, they are fairly reliable players that have proven themselves. To better this model for future analysis, it would be important to take age as a factor as well because younger players tend to be overpaid and older players tend to be underpaid because they are close to retirement. 

In [3]:
# Importing libraries

import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score


# Reading the CSV files (used copilot to help handle special characters that pandas could not read)
nba_performance_22 = pd.read_csv("/workspaces/DS-3021/data/NBA_Perf_22.csv", encoding="ISO-8859-1")
nba_salaries_22 = pd.read_csv("/workspaces/DS-3021/data/nba_salaries_22.csv")

#Drop variables that will not be needed or are duplicates

print(nba_performance_22.info()) # Printing the variables from the dataset

# Dropping unnecessary columns

nba_performance = nba_performance_22.drop(columns = ['Player', 'Tm', 'Pos', 'FGA', 'FG', '3P', '3PA', '2P', '2PA', 'FT', 'FTA'])

# Dropping duplicate variables 

nba_performance = nba_performance.drop_duplicates()

nba_salaries = nba_salaries_22.drop_duplicates()

# Merging the datasets

nba_data = pd.merge(nba_performance_22[['Player']].join(nba_performance), nba_salaries_22, on = 'Player', how =  'inner')

nba_data = nba_data.dropna() # Dropping NA values

# Removing $ from the Salary column

nba_data['Salary'] = nba_data['Salary'].replace(r'[\$,]', '', regex=True)
 # Printing the first five rows of the merged dataframe

nba_data = nba_data.drop(columns=['Player', 'Salary']) # Dropping dependent variables

# Scaling the numeric values in the dataset

scaler = StandardScaler()

nba_scaled = scaler.fit_transform(nba_data) 

nba_df = pd.DataFrame(nba_scaled, columns = nba_data.columns)

#Use the recommended number of cluster (assuming it's different) to retrain your model and visualize the results

kmeans_nba_1 = KMeans(n_clusters = 3, random_state = 42).fit(nba_scaled) # Retraining the model with 3 clusters

# Plotting the visualization

fig = px.scatter_3d(nba_df, x = "DRB", y = "PTS", z = "AST", color=kmeans_nba_1.labels_,
                    title = "Defensive Rebounds vs. Points vs. Assists")
fig.show(renderer = "browser")








<class 'pandas.core.frame.DataFrame'>
RangeIndex: 812 entries, 0 to 811
Data columns (total 29 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Player  812 non-null    object 
 1   Pos     812 non-null    object 
 2   Age     812 non-null    int64  
 3   Tm      812 non-null    object 
 4   G       812 non-null    int64  
 5   GS      812 non-null    int64  
 6   MP      812 non-null    float64
 7   FG      812 non-null    float64
 8   FGA     812 non-null    float64
 9   FG%     797 non-null    float64
 10  3P      812 non-null    float64
 11  3PA     812 non-null    float64
 12  3P%     740 non-null    float64
 13  2P      812 non-null    float64
 14  2PA     812 non-null    float64
 15  2P%     784 non-null    float64
 16  eFG%    797 non-null    float64
 17  FT      812 non-null    float64
 18  FTA     812 non-null    float64
 19  FT%     715 non-null    float64
 20  ORB     812 non-null    float64
 21  DRB     812 non-null    float64
 22  TR

In [4]:
wcss = []
for i in range(1, 11):
    kmeans_basketball = KMeans(n_clusters = i, random_state = 42).fit(nba_scaled)
    wcss.append(kmeans_basketball.inertia_)

elbow_data_nba = pd.DataFrame({"k": range(1, 11), "wcss": wcss})
fig = px.line(elbow_data_nba, x = "k", y = "wcss", title = "Elbow Method for Optimal k")
fig.update_layout(xaxis_title = "Number of Clusters (k)", yaxis_title = "Within-Cluster Sum of Squares (WCSS)")
fig.show(renderer = "browser")